# W4-001: Prepare Black Hole activations (Colab T4)

This notebook creates sentence activation arrays for Week 4 classifier work. It reads the Week 3 split/evidence files and Week 2 state vectors from Google Drive. Sentence activations use only retained earlier sentences plus the target sentence; later sentences are excluded.

## Before running
1. In Colab, select **Runtime → Change runtime type → T4 GPU**.
2. Upload/open this notebook in Colab.
3. Have the updated local file `src/black_hole/dataset.py` ready to upload when prompted. It is needed until that file is available in the GitHub repository.
4. Make sure the Week 2 and Week 3 artifacts are in the `PSSN2` folder in your Google Drive at the paths configured below.

The output is written to Drive after all splits finish. Keep the Colab session active until the final summary appears.

In [ ]:
from pathlib import Path
import subprocess, sys

REPO_URL = 'https://github.com/Rami2212/PSSN-LLM-Reasoning.git'
REPO_DIR = Path('/content/PSSN-LLM-Reasoning')

if not REPO_DIR.exists():
    subprocess.run(['git', 'clone', REPO_URL, str(REPO_DIR)], check=True)

# Install model-extraction libraries without replacing Colab's preinstalled PyTorch.
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q',
                'transformers>=4.51.0,<6', 'accelerate>=1.2.0,<2',
                'safetensors>=0.4.5,<1', 'sentencepiece>=0.2.0,<1'], check=True)

from google.colab import files
print('Upload the updated dataset.py file from your local repository: src/black_hole/dataset.py')
uploaded = files.upload()
if 'dataset.py' not in uploaded:
    raise ValueError('Please upload the file named dataset.py')
destination = REPO_DIR / 'src' / 'black_hole' / 'dataset.py'
destination.parent.mkdir(parents=True, exist_ok=True)
destination.write_bytes(uploaded['dataset.py'])
print('Updated loader installed at', destination)

In [ ]:
from google.colab import drive
import torch

drive.mount('/content/drive')
if not torch.cuda.is_available():
    raise RuntimeError('GPU is not active. Choose Runtime → Change runtime type → T4 GPU, then rerun.')
gpu_name = torch.cuda.get_device_name(0)
print('GPU:', gpu_name)
if 'T4' not in gpu_name.upper():
    raise RuntimeError(f'Expected a Tesla T4, but Colab assigned: {gpu_name}')
print('PyTorch:', torch.__version__, '| CUDA:', torch.version.cuda)

In [ ]:
DRIVE_ROOT = Path('/content/drive/MyDrive/PSSN2')
RUN_DIR = DRIVE_ROOT / 'artifacts' / 'black_hole' / 'w3-1-sentence-3samples-run1'
SPLIT_DIR = RUN_DIR / 'w3-006-splits'
EVIDENCE_DIR = RUN_DIR
STATE_VECTORS = (DRIVE_ROOT / 'artifacts' / 'expanded_baseline' /
    'w2-001-700-attempts-seed44' / 'semantic_state_segmentation' /
    '20261010T145030772067Z' / 'quality_validation' / '20261010T145058972551Z' /
    'representations' / '20261010T145629019430Z' / 'state_vectors.npy')
OUTPUT_DIR = RUN_DIR / 'w4-001-activations'

required = [SPLIT_DIR / 'train.jsonl', SPLIT_DIR / 'train_manifest.json',
            SPLIT_DIR / 'validation.jsonl', SPLIT_DIR / 'validation_manifest.json',
            SPLIT_DIR / 'test.jsonl', SPLIT_DIR / 'test_manifest.json', STATE_VECTORS]
missing = [str(path) for path in required if not path.is_file()]
if missing:
    raise FileNotFoundError('Missing Drive artifact(s):\n' + '\n'.join(missing))
print('Input splits:', SPLIT_DIR)
print('Week 3 evidence:', EVIDENCE_DIR)
print('Week 2 state vectors:', STATE_VECTORS)
print('Output:', OUTPUT_DIR)

## Run extraction
The model revision is read from the Week 3 records and pinned automatically, so activation vectors match the model that produced the labels. The optional Week 2 state vectors are saved separately as `state_features`; they are not mixed into the sentence activation features.

In [ ]:
cmd = [sys.executable, '-m', 'src.black_hole.dataset',
       '--split-dir', str(SPLIT_DIR),
       '--evidence-dir', str(EVIDENCE_DIR),
       '--output-dir', str(OUTPUT_DIR),
       '--state-vectors', str(STATE_VECTORS),
       '--require-cuda']
print('Starting W4-001 activation extraction...')
subprocess.run(cmd, cwd=REPO_DIR, check=True)

In [ ]:
import json, numpy as np

manifest = json.loads((OUTPUT_DIR / 'dataset_manifest.json').read_text())
print(json.dumps(manifest['splits'], indent=2))
for split in ('train', 'validation', 'test'):
    with np.load(OUTPUT_DIR / f'{split}.npz', allow_pickle=False) as data:
        print(split, '| activation features:', data['features'].shape,
              '| labels:', data['labels'].shape,
              '| state baseline:', data['state_features'].shape)
print('Saved to Drive:', OUTPUT_DIR)